<a href="https://colab.research.google.com/github/pySTEPS/ERAD-nowcasting-course-2026/blob/main/notebooks/exercise_notebooks/block_05_blending.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

All exercises can be found in this and the subsequent exercise blocks. The exercises should largely explain themselves and walk you through the different steps to prepare your data and create a nowcast. Note: These exercise notebooks are meant to help you to get started with the exercises. They are meant to be incomplete, so you will have to add some steps yourself (often indicated by the "..." in the code). The number of steps that have to be added by the participants progressively increases the further you get with the exercises. If you need to check your solutions, or if you are looking for the answers, see the [solutions](https://github.com/pySTEPS/ERAD-nowcasting-course-2026/tree/main/notebooks/solutions) notebooks.

# Blending with NWP

In this final block we blend a radar rainfall nowcast with an NWP rainfall forecast, using two of the blending methods in pysteps: [linear blending](https://pysteps.readthedocs.io/en/latest/auto_examples/plot_linear_blending.html) (including its saliency-based variant) and [STEPS blending](https://pysteps.readthedocs.io/en/latest/auto_examples/blended_forecast.html).

A nowcast is skilful for the first hour or so, after which an NWP forecast takes over. Blending combines the two, weighting the nowcast at short lead times and the NWP at longer ones.

We use the Serbian case of 12 August 2017: the RHMSS radar composite and the ECMWF IFS control run initialised at 00 UTC. The forecast is issued at **15:55 UTC** and runs **2 hours** ahead.

Run the helper notebook first. It loads the radar composite and the IFS forecast, puts the NWP data on the radar grid and on the 5-minute nowcast time steps, and does the pre-processing — see [helper_blending.ipynb](helper_blending.ipynb) for the details.

In [ ]:
from google.colab import drive
import os
# mount the Google Drive folder
# don't attempt to remount if the drive is already mounted
if not os.path.exists("/content/mnt/MyDrive"):
  drive.mount("mnt")
%cd '/content/mnt/MyDrive/Colab Notebooks/ERAD-nowcasting-course-2026/notebooks/exercise_notebooks/'
# run the data notebook to load and prepare the radar and NWP data
%run helper_blending.ipynb

## A first look

The IFS runs on a much coarser grid than the radar, and at the issue time the two fields already differ noticeably. The middle panel shows the forecast on its own grid, the right one after the helper reprojected it onto the radar grid.

In [ ]:
# Disable warnings
import warnings
warnings.filterwarnings("ignore")

from matplotlib import pyplot as plt
from pysteps.visualization import plot_precip_field

map_kwargs = {"drawlonlatlines": True}
date_str = f"{date_radar:%Y-%m-%d %H:%M}"
i_native = int(np.searchsorted(nwp_metadata_native["timestamps"], date_radar, side="left"))

plt.figure(figsize=(16, 5))
panels = [(radar_precip[-1], radar_metadata, f"Radar observation at {date_str}"),
          (nwp_precip_native[i_native], nwp_metadata_native, "IFS on its own grid"),
          (nwp_precip[0], nwp_metadata, "IFS on the radar grid")]
for i, (field, geo, title) in enumerate(panels):
    plt.subplot(1, 3, i + 1).set_axis_off()
    plot_precip_field(field, geodata=geo, colorscale="...",  # pick a colorscale
                      title=title, colorbar=(i == 2), map_kwargs=map_kwargs)
plt.tight_layout()
plt.show()

## A helper to look at the results

Every blending method below is inspected in the same way, so we define the plotting and verification once. We use the CRPS as skill score, because it also works for ensemble forecasts.

In [ ]:
from pysteps.verification import probscores

leadtimes_min = [5, 30, 60, 90, 120]


def plot_and_verify(forecast, label):
    """Plot a few lead times against the observations, and plot the CRPS."""
    # accept both deterministic (time, y, x) and ensemble (member, time, y, x)
    ens = forecast if forecast.ndim == 4 else forecast[None]

    plt.figure(figsize=(11, 2.6 * len(leadtimes_min)))
    for n, lt in enumerate(leadtimes_min):
        i = int(lt / timestep) - 1
        panels = [(ens[:, i].mean(axis=0), f"{label} +{lt} min"),
                  (nwp_precip[i], f"IFS +{lt} min"),
                  (precip_obs[i], f"Observation +{lt} min")]
        for col, (field, title) in enumerate(panels):
            plt.subplot(len(leadtimes_min), 3, n * 3 + col + 1).set_axis_off()
            plot_precip_field(field, geodata=radar_metadata, title=title,
                              axis="off", colorbar=False)
    plt.tight_layout()
    plt.show()

    crps_blended, crps_nwp = [], []
    for i in range(n_nowcast_steps):
        crps_blended.append(probscores.CRPS(ens[:, i], precip_obs[i]))
        crps_nwp.append(probscores.CRPS(nwp_precip[i][None], precip_obs[i]))

    fig, ax = plt.subplots(figsize=(9, 4))
    lts = (np.arange(n_nowcast_steps) + 1) * timestep
    ax.plot(lts, crps_nwp, color="tab:blue", lw=2, label="IFS")
    ax.plot(lts, crps_blended, color="tab:orange", lw=2, label=label)
    ax.set_xlabel("Lead time (min)", fontsize=12)
    ax.set_ylabel(r"CRPS (mm h$^{-1}$)", fontsize=12)
    ax.set_title(f"CRPS for the forecast issued at {date_radar:%Y-%m-%d %H:%M} UTC")
    ax.legend(frameon=False, fontsize=12)
    plt.tight_layout()
    plt.show()
    return crps_blended

## Linear blending

The simplest approach: give the nowcast all the weight until `start_blending`, the NWP all the weight from `end_blending` onwards, and interpolate linearly in between. Both times are chosen by the user.

In [ ]:
precip_blended_linear = pysteps.blending.linear_blending.forecast(
    precip=radar_precip_db[-1, :, :],
    precip_metadata=radar_metadata_db,
    velocity=velocity_radar,
    timesteps=n_nowcast_steps,
    timestep=timestep,
    nowcast_method="extrapolation",   # simple advection nowcast
    start_blending=15,   # in minutes: the nowcast stops beating the IFS at +20 min
    end_blending=45,     # in minutes
    precip_nwp=nwp_precip,
    precip_nwp_metadata=nwp_metadata,
)

_ = plot_and_verify(precip_blended_linear, "Linear blending")

## Saliency-based blending

Plain linear blending smooths away the intense cells during the transition. The saliency-based variant preserves pixel intensities that stand out from their surroundings, by ranking them before combining the two forecasts. It is the same function with `saliency=True`.

In [ ]:
# The same call as for the linear blending, but with the saliency option on
precip_blended_salient = ...

_ = plot_and_verify(precip_blended_salient, "Salient blending")

## STEPS blending

Both methods above need the user to pick when the blending starts and ends. The [STEPS blending method](https://pysteps.readthedocs.io/en/latest/pysteps_reference/blending.html) instead derives the weights per spatial scale from the skill of each component, so the transition happens where the data says it should. It also perturbs both components, giving an ensemble.

It needs the NWP forecast in dB, with a leading model dimension, and a motion field for the NWP.

In [ ]:
# Transform the NWP forecast to dB and add the model dimension
nwp_precip_db, nwp_metadata_db = transformer(nwp_precip_steps, nwp_metadata, threshold=0.1)
nwp_precip_db = nwp_precip_db[None, :]

# The motion field of the NWP forecast, one per model and time step
velocity_nwp = []
for n_model in range(nwp_precip_db.shape[0]):
    # Estimate the motion between each pair of consecutive NWP fields
    v = [... for t in range(1, nwp_precip_db.shape[1])]
    # the field at the first time step is the same as at the second
    velocity_nwp.append(np.insert(v, 0, v[0], axis=0))
velocity_nwp = np.stack(velocity_nwp)

precip_blended_steps = pysteps.blending.steps.forecast(
    precip=radar_precip_db,
    precip_models=nwp_precip_db,
    velocity=velocity_radar,
    velocity_models=velocity_nwp,
    timesteps=n_nowcast_steps,
    timestep=timestep,
    issuetime=date_radar,
    n_ens_members=1,     # keep it to 1 member so the notebook stays quick
    precip_thr=radar_metadata_db["threshold"],
    kmperpixel=radar_metadata["xpixelsize"] / 1000.0,
    noise_stddev_adj="auto",
    vel_pert_method=None,
)

# Back to mm/h before verifying
precip_blended_steps, _ = converter(precip_blended_steps, radar_metadata_db)

_ = plot_and_verify(precip_blended_steps, "STEPS blending")

## Things to try

* Move `start_blending` and `end_blending`. The values above were picked by comparing a pure extrapolation nowcast with the IFS lead time by lead time: the nowcast stops winning at +20 min, so the blending is centred there. Can you find a better pair? What happens if you blend far too late, say 60 and 120 minutes?
* Replace `nowcast_method="extrapolation"` with `"steps"` in the linear blending, and pass `nowcast_kwargs={"precip_thr": 0.1, "kmperpixel": 1.0, "timestep": 5, "n_ens_members": 10}`. Does an ensemble nowcast improve the CRPS?
* Raise `n_ens_members` in the STEPS blending (it costs run time, but the CRPS of a single member is pessimistic).
* Blend with the WRF-NMM forecast (`NMM20170812.nc`) instead of the IFS. It is on a finer grid, but only available as 3-hourly accumulations — what does that do to the blended forecast?